# Día 3 - Métricas de clasificación, KNN, SVM y entrada a K-Means

**Fecha:** 15-10-2026

Hoy aprenderemos a responder una pregunta fundamental:

> cuando un clasificador se equivoca, ¿qué tipo de error está cometiendo?

Luego veremos dos familias muy distintas:

- KNN: vecinos y distancias;
- SVM: márgenes y kernels;
- K-Means: primera transición a aprendizaje no supervisado.

## 0. Librerías y carpeta de datos

No hay instalación automática. Si una librería opcional falta, el notebook mostrará el comando que puedes ejecutar manualmente en una celda nueva. Todos los imports principales quedan juntos para que sea claro qué herramientas usa la jornada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.datasets import make_blobs
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.cluster import KMeans

# UCI es opcional; si no está disponible se usa el respaldo local incluido.
try:
    from ucimlrepo import fetch_ucirepo
except ImportError:
    fetch_ucirepo = None

In [ ]:
DATA_DIR = Path('../datos')
if not DATA_DIR.exists():
    DATA_DIR = Path('datos')
print('Datos:', DATA_DIR.resolve())

A partir de aquí, cada bloque de código irá acompañado de una explicación de **qué entra, qué sale y qué debemos mirar**. Ejecutar una celda sin interpretar el resultado no cuenta como comprender el modelo.

# 1. Matriz de confusión: la base de las métricas

Suponiendo que `Sí` es la clase positiva:

| | Real Sí | Real No |
|---|---:|---:|
| Pred Sí | TP | FP |
| Pred No | FN | TN |

- **TP**: predije Sí y era Sí.
- **FP**: predije Sí pero era No.
- **FN**: predije No pero era Sí.
- **TN**: predije No y era No.

La importancia de FP y FN depende del problema de negocio.

Antes de entrenar nada, necesitamos saber **qué tipo de respuesta queremos predecir**.

En clasificación, $y$ no es un número continuo como crecimiento = 2.7. Es una clase, por ejemplo:

- `1` = cliente suscribe depósito;
- `0` = cliente no suscribe.

Un clasificador suele producir primero una **probabilidad o score** y luego una regla convierte ese valor en una clase. Por eso evaluar clasificación requiere mirar **qué clases confunde**.

In [ ]:
# Matriz didáctica del ejemplo de la guía.
TP, FP, FN, TN = 50, 10, 5, 35

accuracy = (TP + TN) / (TP + FP + FN + TN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
specificity = TN / (TN + FP)
f1 = 2 * precision * recall / (precision + recall)

print('Accuracy   :', round(accuracy,3))
print('Precision  :', round(precision,3))
print('Recall     :', round(recall,3))
print('Specificity:', round(specificity,3))
print('F1         :', round(f1,3))

### Cómo leer estos números

Con TP=50, FP=10, FN=5 y TN=35:

- hay **5 positivos reales que el modelo no detectó** → FN;
- hay **10 alarmas positivas que eran negativas** → FP.

Cuál error importa más depende del problema. En riesgo financiero, salud o fraude, perder un positivo real puede tener un costo muy distinto a generar una falsa alarma.

Las fórmulas son:

$$Accuracy=\frac{TP+TN}{TP+TN+FP+FN}$$

$$Precision=\frac{TP}{TP+FP}$$

$$Recall=\frac{TP}{TP+FN}$$

$$F1=2\frac{Precision\cdot Recall}{Precision+Recall}$$

No existe una métrica universalmente mejor.

## 1.1 ¿Qué responde cada métrica?

- Accuracy: ¿qué proporción total acerté?
- Precision: de lo que marqué positivo, ¿cuánto era realmente positivo?
- Recall: de todos los positivos reales, ¿cuántos encontré?
- F1: equilibrio armónico entre Precision y Recall.

No hay una métrica universalmente mejor. Depende del costo de errores.

| Métrica | Pregunta operativa | Puede engañar cuando... |
|---|---|---|
| Accuracy | ¿qué proporción total acerté? | las clases están muy desbalanceadas |
| Precision | de lo que marqué positivo, ¿cuánto era positivo? | ignoramos positivos que nunca detectamos |
| Recall | de todos los positivos reales, ¿cuántos encontré? | generamos demasiadas falsas alarmas |
| F1 | ¿equilibrio Precision/Recall? | los costos reales no son simétricos |
| ROC-AUC | ¿ordena positivos por encima de negativos? | la clase positiva es muy rara |
| PR-AUC | ¿cómo se comporta Precision/Recall a muchos thresholds? | necesitamos además entender calibración/costos |

## 1.2 Accuracy puede ocultar un modelo inútil

Imagina 1000 casos donde solo 50 son positivos. Un clasificador que diga `0` a todos tiene 95% Accuracy y 0% Recall para la clase que nos interesa.

In [ ]:
y_imb=np.array([1]*50+[0]*950)
pred_todos_cero=np.zeros(1000,dtype=int)

print('Accuracy:',accuracy_score(y_imb,pred_todos_cero))
print('Recall  :',recall_score(y_imb,pred_todos_cero,zero_division=0))
print('F1      :',f1_score(y_imb,pred_todos_cero,zero_division=0))

## 1.3 Specificity y clase positiva

La “clase positiva” no significa “buena”. Significa el evento de interés para el análisis.

Si estudiamos riesgo de impago, podríamos definir `impago=1`. Entonces Recall responde: **¿qué proporción de los impagos reales detectamos?**

# 2. Threshold: una probabilidad no es todavía una clase

Muchos clasificadores producen probabilidades o scores. Para convertirlos a clase usamos un umbral.

Si bajamos el threshold:

- detectamos más positivos -> Recall suele subir;
- también generamos más falsos positivos -> Precision puede bajar.

Supón que el modelo produce $P(y=1)=0.62$. Esa probabilidad todavía no es una clase.

Con threshold 0.50:

$$0.62 \ge 0.50 \Rightarrow \hat y=1$$

Si elevamos el threshold a 0.80, ese mismo caso pasa a 0. Por tanto **mover threshold no reentrena el modelo**: cambia la decisión final sobre sus probabilidades.

En general:

- threshold menor → más positivos predichos → suele subir Recall y bajar Precision;
- threshold mayor → menos positivos predichos → suele subir Precision y bajar Recall.

In [ ]:
# Ejemplo artificial de probabilidades predichas.
y_real_demo = np.array([1,1,1,0,0,0,1,0,1,0])
prob = np.array([.95,.72,.55,.62,.45,.20,.48,.35,.81,.10])

for umbral in [0.30,0.50,0.70]:
    pred = (prob >= umbral).astype(int)
    print()
    print('Threshold =', umbral)
    print('Precision:', round(precision_score(y_real_demo,pred,zero_division=0),2))
    print('Recall   :', round(recall_score(y_real_demo,pred,zero_division=0),2))
    print('F1       :', round(f1_score(y_real_demo,pred,zero_division=0),2))

## 2.1 Precision-Recall al mover el threshold

En vez de probar solo tres umbrales, podemos construir una tabla completa.

In [ ]:
filas=[]
for umbral in np.arange(.1,.91,.1):
    p=(prob>=umbral).astype(int)
    filas.append({
        'threshold':umbral,
        'precision':precision_score(y_real_demo,p,zero_division=0),
        'recall':recall_score(y_real_demo,p,zero_division=0),
        'f1':f1_score(y_real_demo,p,zero_division=0)
    })

pd.DataFrame(filas).round(2)

No “optimizamos” el threshold mirando Test. El threshold forma parte de las decisiones del modelo y se escoge usando Validation, idealmente considerando costos de negocio.

# 3. Recuperar Bank Marketing y mantener el mismo problema

Una buena práctica de aprendizaje es usar el mismo dataset para comparar varios modelos. Así las diferencias no se deben a que cambiamos de problema.

## Antes de modelar: entender Bank Marketing

El objetivo es predecir si un cliente suscribirá un depósito a plazo (`y`). Antes de usar KNN o SVM debemos revisar:

- qué representa cada fila;
- tipos numéricos/categóricos;
- distribución de `y`;
- nulos o categorías `unknown`;
- si alguna variable estará disponible **en el momento real de predecir**.

Ese último punto es crítico: una columna puede existir en el CSV y aun así ser leakage para el uso operativo previsto.

In [ ]:
if fetch_ucirepo is not None:
    try:
        bank = fetch_ucirepo(id=222)
        df_bank = pd.concat([bank.data.features, bank.data.targets], axis=1)
        fuente = 'UCI'
    except Exception:
        df_bank = pd.read_csv(DATA_DIR/'bank_marketing_fallback.csv')
        fuente = 'Respaldo sintético'
else:
    df_bank = pd.read_csv(DATA_DIR/'bank_marketing_fallback.csv')
    fuente = 'Respaldo sintético'

print('Fuente:', fuente)
datos = df_bank.drop(columns=['tipo_dato'], errors='ignore').copy()
y = datos['y'].astype(str).str.lower().map({'yes':1,'no':0})
X = pd.get_dummies(datos.drop(columns=['y']), drop_first=True, dtype=int)

X_train_val, X_test, y_train_val, y_test = train_test_split(X,y,test_size=.15,stratify=y,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val,y_train_val,test_size=.1765,stratify=y_train_val,random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)

## 3.1 Mantener un baseline y una tabla común

Todos los modelos de hoy se compararán sobre el mismo Validation y con las mismas observaciones. Esa disciplina hace que las métricas sean comparables.

### Regresión logística como baseline lineal de clasificación

Aunque el contenido central del día sea KNN/SVM, conviene tener un baseline interpretable. La regresión logística calcula un score lineal:

$$
z=b_0+b_1x_1+\dots+b_px_p
$$

y lo transforma con la sigmoide:

$$
P(y=1)=\frac{1}{1+e^{-z}}
$$

Así obtenemos probabilidades entre 0 y 1. Esto será una referencia para saber si los modelos más flexibles realmente aportan.

In [ ]:
# Preprocesamiento común: escalar numéricas y one-hot para categorías.
num_cols_logit = [c for c in ['age','balance','duration','campaign','pdays','previous'] if c in df_bank.columns]
cat_cols_logit = [c for c in ['job','marital','education','housing','loan','poutcome'] if c in df_bank.columns]

X_logit = df_bank[num_cols_logit + cat_cols_logit].copy()
y_logit = (df_bank['y'].astype(str).str.lower().isin(['yes','1','true'])).astype(int)

# Usamos nombres separados para NO sobrescribir los conjuntos que luego usará KNN/SVM.
X_train_logit, X_temp_logit, y_train_logit, y_temp_logit = train_test_split(
    X_logit, y_logit, test_size=.30, random_state=42, stratify=y_logit
)
X_val_logit, X_test_logit, y_val_logit, y_test_logit = train_test_split(
    X_temp_logit, y_temp_logit, test_size=.50, random_state=42, stratify=y_temp_logit
)

pre_logit = ColumnTransformer([
    ('num', StandardScaler(), num_cols_logit),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_logit)
])

logit = Pipeline([
    ('pre', pre_logit),
    ('model', LogisticRegression(max_iter=3000, class_weight='balanced'))
])

logit.fit(X_train_logit, y_train_logit)
prob_logit = logit.predict_proba(X_val_logit)[:,1]
pred_logit = (prob_logit >= .5).astype(int)

print(classification_report(y_val_logit, pred_logit, digits=3, zero_division=0))

### Cómo leer `classification_report`

El reporte separa métricas por clase. `support` es cuántas observaciones reales pertenecen a cada clase.

No te quedes solo con `accuracy`. Si `y=1` es minoritaria, revisa especialmente su Recall, Precision y F1. `macro avg` da igual peso a ambas clases; `weighted avg` pondera por cantidad de casos.

In [ ]:
clase_may=int(y_train.mode().iloc[0])
p_base=np.repeat(clase_may,len(y_val))

resultados_modelos={
    'Baseline_mayoria':{
        'Accuracy':accuracy_score(y_val,p_base),
        'Precision':precision_score(y_val,p_base,zero_division=0),
        'Recall':recall_score(y_val,p_base,zero_division=0),
        'F1':f1_score(y_val,p_base,zero_division=0)
    }
}
pd.DataFrame(resultados_modelos).T.round(3)

# 4. KNN: el modelo de los vecinos

KNN no aprende una ecuación. Para clasificar un caso nuevo:

1. mide su distancia con observaciones de Train;
2. encuentra los `k` vecinos más cercanos;
3. hace una votación.

Por eso la escala de las variables importa muchísimo.

KNN no aprende una ecuación. **Guarda el conjunto de entrenamiento** y, cuando llega un caso nuevo, busca los casos más cercanos.

Para un punto $x$:

1. calculamos distancia contra observaciones de Train;
2. escogemos los $k$ vecinos más cercanos;
3. en clasificación, votan por la clase.

Dos distancias comunes:

$$d_{euclidiana}=\sqrt{\sum_j(x_j-z_j)^2}$$

$$d_{manhattan}=\sum_j|x_j-z_j|$$

La distancia hace que la **escala** sea crucial.

## 4.1 Euclidiana y Manhattan

Para dos puntos A=(2,70) y B=(5,74):

\[
d_{euclidiana}=\sqrt{(5-2)^2+(74-70)^2}=5
\]

\[
d_{manhattan}=|5-2|+|74-70|=7
\]

In [ ]:
A = np.array([2,70])
B = np.array([5,74])

euclidiana = np.sqrt(np.sum((A-B)**2))
manhattan = np.sum(np.abs(A-B))

print('Euclidiana:', euclidiana)
print('Manhattan :', manhattan)

## 4.1.1 KNN manual con cinco puntos

Primero calculemos distancias sin usar `KNeighborsClassifier`.

In [ ]:
# Dos features sencillas: [horas, asistencia].
X_tiny=np.array([[2,65],[3,70],[5,82],[6,88],[8,92]],dtype=float)
y_tiny=np.array([0,0,1,1,1])
nuevo=np.array([4,78],dtype=float)

# Distancia euclidiana desde el nuevo caso hacia cada observación.
dist=np.sqrt(np.sum((X_tiny-nuevo)**2,axis=1))
orden=np.argsort(dist)

pd.DataFrame({'horas':X_tiny[:,0],'asistencia':X_tiny[:,1],'clase':y_tiny,'distancia':dist}).sort_values('distancia')

In [ ]:
# Con k=3 tomamos las clases de los tres vecinos más cercanos.
k=3
vecinos=y_tiny[orden[:k]]
pred_manual=int(np.mean(vecinos)>=0.5)
print('Vecinos:',vecinos)
print('Predicción manual:',pred_manual)

## 4.2 Por qué escalamos

Si `balance` llega a 10000 y `age` llega a 90, la distancia puede quedar dominada por balance solo por su unidad de medida.

`StandardScaler` transforma cada variable aproximadamente a media 0 y desviación 1 usando **solo Train**.

Si `balance` va de -2000 a 50000 y `campaign` de 1 a 10, la primera variable dominaría la distancia aunque no sea más importante. `StandardScaler` transforma cada variable usando la media y desviación de Train:

$$
z=\frac{x-\mu_{train}}{\sigma_{train}}
$$

No debemos calcular media/desviación usando Validation/Test, porque introduciríamos información externa al entrenamiento.

In [ ]:
scaler = StandardScaler()

# fit_transform en TRAIN: aprende media/std y transforma.
X_train_s = scaler.fit_transform(X_train)

# transform en Validation/Test: reutiliza los parámetros aprendidos en Train.
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

In [ ]:
resultados_knn=[]

for k in [1,3,5,9,15,25,41]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    p = knn.predict(X_val_s)
    resultados_knn.append({
        'k':k,
        'Accuracy':accuracy_score(y_val,p),
        'Precision':precision_score(y_val,p,zero_division=0),
        'Recall':recall_score(y_val,p,zero_division=0),
        'F1':f1_score(y_val,p,zero_division=0)
    })

pd.DataFrame(resultados_knn).round(3)

`k=1` puede memorizar demasiado. Un `k` enorme suaviza tanto que puede perder estructuras útiles. `k` es un hiperparámetro y se selecciona con Validation/CV.

## 4.3 KNN con pesos por distancia

Con `weights='distance'`, un vecino muy cercano pesa más que otro que apenas entra dentro de los K seleccionados.

In [ ]:
for pesos in ['uniform','distance']:
    m=KNeighborsClassifier(n_neighbors=9,weights=pesos)
    m.fit(X_train_s,y_train)
    p=m.predict(X_val_s)
    print(pesos,'-> F1:',round(f1_score(y_val,p,zero_division=0),3))

## 4.4 Euclidiana vs Manhattan en sklearn

Con `metric='minkowski', p=2` obtenemos Euclidiana; con `p=1`, Manhattan.

In [ ]:
for p_dist in [1,2]:
    m=KNeighborsClassifier(n_neighbors=9,metric='minkowski',p=p_dist)
    m.fit(X_train_s,y_train)
    pred=m.predict(X_val_s)
    print('p=',p_dist,'F1=',round(f1_score(y_val,pred,zero_division=0),3))

# 5. SVM: separar clases con un margen

SVM busca una frontera que separe clases intentando mantener un margen amplio.

Parámetros importantes:

- `C`: cuánto penalizamos errores;
- `kernel`: forma de similitud (`linear`, `rbf`, ...);
- `gamma` en RBF: qué tan local es la influencia de cada punto.

SVM también es sensible al escalado.

Una SVM busca una frontera que separe clases dejando un **margen** amplio.

Los puntos más importantes son los **support vectors**: observaciones cercanas al margen o que lo violan.

- `C` grande penaliza fuertemente errores → frontera más rígida/flexible respecto a casos difíciles;
- `C` pequeño acepta más violaciones → mayor regularización;
- `kernel='linear'` crea una frontera lineal;
- `kernel='rbf'` permite fronteras no lineales;
- `gamma` alto hace que cada punto tenga influencia más local; demasiado alto puede sobreajustar.

SVM también es muy sensible a escala, por eso normalmente la acompañamos de `StandardScaler`.

In [ ]:
# Usamos una muestra si el dataset es grande para que la clase sea ágil.
max_train = min(5000, len(X_train_s))
idx = np.random.default_rng(42).choice(len(X_train_s), size=max_train, replace=False)

Xtr_svm = X_train_s[idx]
ytr_svm = y_train.iloc[idx]

svm = SVC(
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=True,
    random_state=42
)
svm.fit(Xtr_svm, ytr_svm)

pred_svm = svm.predict(X_val_s)
prob_svm = svm.predict_proba(X_val_s)[:,1]

print('F1:', round(f1_score(y_val,pred_svm),3))
print('ROC-AUC:', round(roc_auc_score(y_val,prob_svm),3))

## 5.0 Intuición visual con un SVM lineal

En dos dimensiones podemos visualizar la frontera y los *support vectors*. Los support vectors son puntos especialmente relevantes para definir el margen.

In [ ]:

X_vis,y_vis=make_blobs(n_samples=80,centers=2,cluster_std=1.25,random_state=7)
svm_lin=SVC(kernel='linear',C=1).fit(X_vis,y_vis)

plt.figure(figsize=(7,5))
plt.scatter(X_vis[:,0],X_vis[:,1],c=y_vis)
plt.scatter(
    svm_lin.support_vectors_[:,0],svm_lin.support_vectors_[:,1],
    s=180,facecolors='none',edgecolors='black',label='Support vectors'
)
plt.legend()
plt.title('Los puntos marcados ayudan a definir el margen')
plt.show()

## 5.1 Mover C y gamma

Prueba diferentes combinaciones. No elijas por Train: mira Validation.

### Qué debes mirar al mover hiperparámetros

No busques solo el mayor F1. Compara Train y Validation. Un patrón típico de exceso de flexibilidad es:

- Train casi perfecto;
- Validation deja de mejorar o empeora.

Eso indica que aumentar complejidad ya está capturando particularidades de Train.

In [ ]:
experimentos=[]

for C in [0.1,1,10]:
    for gamma in ['scale',0.01,0.1]:
        m = SVC(C=C,kernel='rbf',gamma=gamma)
        m.fit(Xtr_svm,ytr_svm)
        p = m.predict(X_val_s)
        experimentos.append({'C':C,'gamma':str(gamma),'F1_val':f1_score(y_val,p,zero_division=0)})

pd.DataFrame(experimentos).sort_values('F1_val',ascending=False).round(3)

## 5.2 Kernel lineal vs RBF

Un kernel lineal busca una frontera lineal. RBF puede construir fronteras no lineales. Más flexibilidad no significa automáticamente mejor generalización.

In [ ]:
comparar=[]
for kernel in ['linear','rbf']:
    m=SVC(C=1,kernel=kernel,gamma='scale')
    m.fit(Xtr_svm,ytr_svm)
    p=m.predict(X_val_s)
    comparar.append({'kernel':kernel,'F1_val':f1_score(y_val,p,zero_division=0)})

pd.DataFrame(comparar).round(3)

## 5.3 Crear una tabla común KNN vs SVM

In [ ]:
knn_final=KNeighborsClassifier(n_neighbors=9,weights='distance')
knn_final.fit(X_train_s,y_train)
p_knn=knn_final.predict(X_val_s)

pd.DataFrame({
    'KNN':{
        'Accuracy':accuracy_score(y_val,p_knn),
        'Precision':precision_score(y_val,p_knn,zero_division=0),
        'Recall':recall_score(y_val,p_knn,zero_division=0),
        'F1':f1_score(y_val,p_knn,zero_division=0)
    },
    'SVM':{
        'Accuracy':accuracy_score(y_val,pred_svm),
        'Precision':precision_score(y_val,pred_svm,zero_division=0),
        'Recall':recall_score(y_val,pred_svm,zero_division=0),
        'F1':f1_score(y_val,pred_svm,zero_division=0)
    }
}).T.round(3)

# 6. Curvas ROC y Precision-Recall

ROC-AUC resume capacidad de ranking a distintos thresholds. PR-AUC suele ser especialmente informativa cuando la clase positiva es rara.

Las curvas usan **scores/probabilidades**, no una única clasificación fija.

- ROC compara True Positive Rate con False Positive Rate a muchos thresholds.
- Precision-Recall se concentra directamente en la calidad/cobertura de la clase positiva y suele ser más informativa con clases raras.

AUC resume capacidad de ranking, pero no decide por nosotros qué threshold usar en producción.

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
RocCurveDisplay.from_predictions(y_val, prob_svm, ax=ax)
plt.title('SVM - ROC en Validation')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
PrecisionRecallDisplay.from_predictions(y_val, prob_svm, ax=ax)
plt.title('SVM - Precision/Recall en Validation')
plt.show()

## 6.1 ROC-AUC no sustituye la matriz de confusión

AUC resume ranking a muchos thresholds, pero una decisión operativa termina usando un threshold concreto. Por eso conviene leer ambas cosas: capacidad global de ranking y errores en el punto de operación elegido.

# 7. Primer paso a aprendizaje no supervisado: K-Means

Hasta ahora siempre teníamos `y`. En clustering **no existe target conocido**.

La pregunta cambia de:

> “¿qué clase tiene este registro?”

A:

> “¿qué observaciones se parecen entre sí?”

K-Means intenta encontrar `k` centroides y asigna cada observación al centroide más cercano.

Hasta aquí existía una `y` conocida. K-Means cambia la pregunta: **no hay respuesta correcta entregada al algoritmo**.

Ya no preguntamos “¿predije yes/no?”, sino “¿existen grupos de observaciones parecidas?”. Por eso desaparecen Accuracy/F1 y aparecen criterios como inercia, silhouette, estabilidad e interpretación de clusters.

In [ ]:
# Usamos solo 3 variables numéricas para poder visualizar mejor la idea.
cols_cluster = ['age','balance','duration']
cols_disponibles = [c for c in cols_cluster if c in datos.columns]

X_cluster = datos[cols_disponibles].select_dtypes(include='number').dropna()
scaler_cluster = StandardScaler()
Xc = scaler_cluster.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(Xc)

pd.Series(labels).value_counts().sort_index()

## 7.1 Un K-Means manual en 2D

Para construir intuición, generamos tres grupos visibles. El algoritmo no recibe las etiquetas verdaderas; solo coordenadas.

In [ ]:

X_blobs,_=make_blobs(n_samples=120,centers=3,cluster_std=.75,random_state=12)
km_demo=KMeans(n_clusters=3,n_init=20,random_state=42)
lab_demo=km_demo.fit_predict(X_blobs)

plt.figure(figsize=(7,5))
plt.scatter(X_blobs[:,0],X_blobs[:,1],c=lab_demo,alpha=.7)
plt.scatter(km_demo.cluster_centers_[:,0],km_demo.cluster_centers_[:,1],s=220,marker='X',label='Centroides')
plt.legend()
plt.title('K-Means encuentra centroides sin conocer una y')
plt.show()

Observa que los números 0, 1 y 2 de los clusters son etiquetas arbitrarias. No tienen orden ni significado hasta que describimos cada grupo.

Mañana profundizaremos en:

- cómo elegir K;
- elbow e inertia;
- silhouette;
- clustering jerárquico;
- interpretación económica de segmentos.

# Cierre y preguntas

1. ¿Por qué Accuracy puede ser engañosa?
2. ¿Qué diferencia existe entre Precision y Recall?
3. ¿Por qué KNN necesita escalado?
4. ¿Qué controla `k`?
5. ¿Qué controla `C` en SVM?
6. ¿Qué significa pasar de aprendizaje supervisado a no supervisado?

## Qué deberías poder explicar al cerrar el Día 3

- por qué Accuracy sola puede engañar;
- diferencia FP vs FN en lenguaje del negocio;
- cómo threshold modifica Precision/Recall;
- por qué KNN necesita escala y qué significa `k`;
- diferencia Euclidiana vs Manhattan;
- qué son margen y support vectors en SVM;
- efecto general de `C` y `gamma`;
- por qué ROC-AUC no reemplaza una matriz de confusión;
- por qué K-Means ya pertenece a aprendizaje no supervisado.